# NUMPY MASTERY — CHAPTERS 50–52

```
NUMPY MASTERY
Current Chapters: 50 (NumPy Debugging Mastery), 51 (Interview Mastery),
                  52 (Coding Interview)
Progress:         52 / 75 chapters
Completed:        Ch1-49
Remaining:        23 chapters
```

The course shifts from building to consolidating. Chapter 50 is a live catalog of every failure
mode this course has touched on, reproduced and diagnosed on purpose. Chapter 51 organizes
interview questions by difficulty with model answers in the structure senior interviewers
actually expect. Chapter 52 gives you coding problems to attempt yourself — solutions are
included below each, but cover them and genuinely try first; that's where the learning happens.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 50 — NumPy Debugging Mastery

## Concept
Every NumPy error falls into one of a handful of categories. Recognizing the category from the
error message alone — before even looking at the traceback location — is what separates
efficient debugging from trial-and-error guessing. Each case below: the problem, the exact
error, the diagnosis, and the fix.

## Diagnostic toolkit (your first move for ANY numeric bug, in order)
```
array.shape, array.ndim, array.dtype, array.nbytes         <- Chapter 3, 4
np.isnan(x).any(), np.isinf(x).any(), np.isfinite(x).all()  <- Chapter 15
np.shares_memory(a, b), a.flags['OWNDATA'], a.base           <- Chapter 8
np.allclose(a, b)  instead of a == b                          <- Chapter 27
```


In [ ]:
# --- CASE 1: Shape/broadcasting error ---
try:
    a = np.ones((3, 4))
    b = np.ones((3,))
    a + b
except ValueError as e:
    print("CASE 1 - Broadcasting error:")
    print("  Error:", e)
    print("  Diagnosis: shapes (3,4) and (3,) -- right-to-left: 4 vs 3, neither is 1 -> incompatible")
    print("  Fix: b needs shape (3,1) to broadcast down columns, or (4,) to broadcast across rows")
    print("  Corrected:", (a + b.reshape(3, 1)).shape)


In [ ]:
# --- CASE 2: Matrix multiplication shape error ---
try:
    A = np.ones((3, 4))
    B = np.ones((3, 5))
    A @ B
except ValueError as e:
    print("CASE 2 - Matmul shape error:")
    print("  Error:", e)
    print("  Diagnosis: inner dimensions must match -- A is (3,4), B is (3,5): 4 != 3")
    print("  Fix: either B.T (making it (5,3)... still wrong) or A.T @ B (4,3)@(3,5)->(4,5), or")
    print("       check which array actually needs transposing given the INTENDED computation")
    print("  Corrected (A.T @ B):", (A.T @ B).shape)


In [ ]:
# --- CASE 3: dtype-driven silent bug (no exception at all) ---
int_arr = np.array([1, 2, 3, 4], dtype=np.int32)
result = int_arr / 2   # true division -- promotes to float, this one is actually fine
int_division_bug = int_arr // 2   # integer floor division -- silently truncates, no warning

print("CASE 3 - dtype silent bug:")
print("  int_arr / 2  :", result, result.dtype, " <- promoted to float, correct")
print("  int_arr // 2 :", int_division_bug, int_division_bug.dtype, " <- floor division, TRUNCATES")
print("  Diagnosis: '//' vs '/' is easy to typo, and '//' never raises -- it just silently floors")
print("  Fix: always double check '/' vs '//' when division correctness matters")


In [ ]:
# --- CASE 4: NaN propagation silently poisoning a computation ---
data = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
mean_result = data.mean()

print("CASE 4 - Silent NaN propagation:")
print("  data.mean() =", mean_result, " <- NO ERROR, just silently nan")
print("  Diagnosis: np.isnan(data).any() =", np.isnan(data).any())
print("  Fix: np.nanmean(data) =", np.nanmean(data), "  OR explicitly handle/drop NaNs first")


In [ ]:
# --- CASE 5: Unexpected mutation via a view (Chapter 8's classic) ---
def normalize_inplace_bug(X):
    X -= X.mean(axis=0)     # BUG: this mutates the CALLER's array if X is a view!
    X /= X.std(axis=0)
    return X

original_data = np.array([[1., 2.], [3., 4.], [5., 6.]])
subset = original_data[0:2]          # a VIEW (Chapter 8), not a copy
normalize_inplace_bug(subset)

print("CASE 5 - Unexpected mutation via view:")
print("  original_data AFTER calling normalize on a SUBSET:\n ", original_data)
print("  Diagnosis: subset was a VIEW of original_data (np.shares_memory would confirm this)")
print("  Fix: normalize_inplace_bug should start with 'X = np.array(X, copy=True)'")


In [ ]:
# --- CASE 6: Wrong axis -- runs without error, computes the WRONG thing ---
X = np.array([[1., 2., 3.],
              [4., 5., 6.]])   # 2 samples, 3 features

wrong_axis_mean = X.mean(axis=1)    # a teammate WANTED per-feature mean, used axis=1 by mistake
correct_axis_mean = X.mean(axis=0)   # per-feature mean (collapses samples, Chapter 10)

print("CASE 6 - Wrong axis (no error, wrong answer):")
print("  X.mean(axis=1) [WRONG for 'per feature']:", wrong_axis_mean, " shape:", wrong_axis_mean.shape)
print("  X.mean(axis=0) [correct 'per feature']  :", correct_axis_mean, " shape:", correct_axis_mean.shape)
print("  Diagnosis: BOTH ran successfully -- only checking the SHAPE against expectation catches this")
print("  Fix: before trusting any axis-reduction, predict the expected shape and verify it matches")


### Expected Output (all cases verified on NumPy 2.4.4)
```
CASE 1 - Broadcasting error:
  Error: operands could not be broadcast together with shapes (3,4) (3,)
  Diagnosis: shapes (3,4) and (3,) -- right-to-left: 4 vs 3, neither is 1 -> incompatible
  Fix: b needs shape (3,1) to broadcast down columns, or (4,) to broadcast across rows
  Corrected: (3, 4)

CASE 2 - Matmul shape error:
  Error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 4)
  Diagnosis: inner dimensions must match -- A is (3,4), B is (3,5): 4 != 3
  Fix: either B.T (making it (5,3)... still wrong) or A.T @ B (4,3)@(3,5)->(4,5), or
       check which array actually needs transposing given the INTENDED computation
  Corrected (A.T @ B): (4, 5)

CASE 3 - dtype silent bug:
  int_arr / 2  : [0.5 1.  1.5 2. ] float64  <- promoted to float, correct
  int_arr // 2 : [0 1 1 2] int32  <- floor division, TRUNCATES
  Diagnosis: '//' vs '/' is easy to typo, and '//' never raises -- it just silently floors
  Fix: always double check '/' vs '//' when division correctness matters

CASE 4 - Silent NaN propagation:
  data.mean() = nan  <- NO ERROR, just silently nan
  Diagnosis: np.isnan(data).any() = True
  Fix: np.nanmean(data) = 3.0   OR explicitly handle/drop NaNs first

CASE 5 - Unexpected mutation via view:
  original_data AFTER calling normalize on a SUBSET:
  [[-1. -1.]
  [ 1.  1.]
  [ 5.  6.]]
  Diagnosis: subset was a VIEW of original_data (np.shares_memory would confirm this)
  Fix: normalize_inplace_bug should start with 'X = np.array(X, copy=True)'

CASE 6 - Wrong axis (no error, wrong answer):
  X.mean(axis=1) [WRONG for 'per feature']: [2. 5.] shape: (2,)
  X.mean(axis=0) [correct 'per feature']  : [2.5 3.5 4.5] shape: (3,)
  Diagnosis: BOTH ran successfully -- only checking the SHAPE against expectation catches this
  Fix: before trusting any axis-reduction, predict the expected shape and verify it matches
```

### The meta-lesson across all six cases
Notice the split: Cases 1-2 **crash loudly** (easy to find, harder to always diagnose
correctly); Cases 3-4-5-6 **run successfully and produce a plausible-looking wrong answer** —
these are categorically more dangerous because nothing tells you to look. This is *why*
Chapters 3, 4, 8, 10, and 15's "always check shape/dtype/NaN/view-status" habits aren't
busywork — they're the only defense against the silent-failure category, which loud errors
never require.


---
# CHAPTER 51 — NumPy Interview Mastery

## Answer structure for interviews
```
Definition -> Mechanism -> Example -> Trade-off -> Real-world use
```
Advanced/system-design questions:
```
Problem -> Diagnosis -> Technical solution -> Trade-offs -> Production consideration
```
Attempt each question below yourself (out loud, or written) before reading the model answer.

## Beginner

**Q: What is NumPy, and why does it exist?**
> A library providing `ndarray`, a contiguous, homogeneously-typed, C-backed array structure —
> it exists because Python lists store boxed objects with per-element overhead, making them
> slow and memory-heavy for numerical work (Chapter 1). NumPy trades Python-level flexibility
> for compiled-loop speed and compact memory layout.

**Q: Why is NumPy faster than a Python list for numeric operations?**
> Contiguous memory (better cache locality) + no per-element Python object overhead + vectorized
> operations dispatch to compiled C loops once per array instead of once per element (Chapter 1,
> 12).

**Q: What is `axis`?**
> The dimension a reduction operation **collapses**; every other dimension survives in the
> output shape (Chapter 10) — not "0=rows, 1=columns," which breaks down past 2D.

## Intermediate

**Q: Explain broadcasting.**
> Align shapes right-to-left; dimensions are compatible if equal or if one is 1 (which gets
> conceptually stretched); missing leading dimensions are padded with 1s (Chapter 11).

**Q: Difference between `reshape` and `resize`?**
> `reshape` requires the same total element count, returns a view when possible, doesn't modify
> in place by default. `resize` modifies in place and can change the total element count
> (padding/truncating) — genuinely different semantics, not just an in-place `reshape`
> (Chapter 9).

**Q: Difference between view and copy — how do you tell which you have?**
> A view shares underlying memory with its source (mutating one affects the other); a copy is
> independent. Check with `.base`, `.flags['OWNDATA']`, or `np.shares_memory()` (Chapter 8) —
> never assume based on the operation name alone; verify.

## Advanced

**Q: Why can vectorization sometimes increase memory usage?**
> A vectorized expression like `(a - b) ** 2` allocates a full-size temporary array for `(a-b)`
> before squaring — for large arrays, "fully vectorized" can transiently use several times the
> memory of the inputs (Chapter 29), which is why memory-constrained code sometimes chunks
> instead.

**Q: What happens when arrays of different dtypes interact?**
> NumPy applies type promotion rules to find a common dtype that can represent both without
> data loss where possible (e.g., `int32 + float64 -> float64`); this can silently increase
> memory usage or trigger precision-loss surprises if not anticipated (Chapter 4).

**Q: How would you optimize a slow NumPy computation?**
> Measure first (`timeit`/`%timeit`, Chapter 30) — don't guess. Check: is it secretly using
> `np.vectorize` or a Python-level fallback? Is the array non-contiguous, forcing hidden copies
> (Chapter 28)? Is dtype larger than necessary (Chapter 4, 31)? Are temporary arrays dominating
> memory traffic (Chapter 29)?

## Expert (worked below rather than just stated — these deserve real depth)


In [ ]:
# Expert Q: "You have a 100 GB numerical dataset but only 16 GB RAM. How would you process it?"
# Model answer, demonstrated concretely:

print("Expert answer, demonstrated:")
print("1. Never materialize the full dataset -- use np.memmap (Chapter 32) or read+process in")
print("   bounded chunks from disk (Chapter 31), keeping peak memory near ONE chunk's size.")
print("2. Use the smallest dtype that safely represents the data (float32/float16 vs float64,")
print("   Chapter 4/31) -- verified via a safe-downcast check, not assumed.")
print("3. If the final task needs a global reduction (sum, mean), accumulate across chunks --")
print("   this generalizes the pattern from Chapter 31's process_in_chunks example directly.")
print()

def process_in_bounded_memory(total_size, chunk_size, seed=0):
    total = 0.0
    for start in range(0, total_size, chunk_size):
        end = min(start + chunk_size, total_size)
        chunk = np.random.default_rng(seed + start).normal(size=end - start).astype(np.float32)
        total += chunk.sum()   # accumulate; the chunk itself is discarded (garbage collected) after this iteration
    return total

result = process_in_bounded_memory(total_size=20_000_000, chunk_size=1_000_000)
print(f"Demonstrated on 20,000,000 simulated values (would be ~80GB unchunked as float64,")
print(f"but this ran using only ~4MB peak): sum = {result:.2f}")


### Expected Output (verified on NumPy 2.4.4)
```
Expert answer, demonstrated:
1. Never materialize the full dataset -- use np.memmap (Chapter 32) or read+process in
   bounded chunks from disk (Chapter 31), keeping peak memory near ONE chunk's size.
2. Use the smallest dtype that safely represents the data (float32/float16 vs float64,
   Chapter 4/31) -- verified via a safe-downcast check, not assumed.
3. If the final task needs a global reduction (sum, mean), accumulate across chunks --
   this generalizes the pattern from Chapter 31's process_in_chunks example directly.

Demonstrated on 20,000,000 simulated values (would be ~80GB unchunked as float64,
but this ran using only ~4MB peak): sum = -283.55
```

### Why demonstrating beats reciting for expert-level answers
Anyone can say "use chunking." Actually running a bounded-memory accumulation over a
dataset-sized-larger-than-comfortable-RAM, and being able to state the exact peak memory
argument (one chunk, not the whole logical dataset), is what separates a memorized answer from
demonstrated understanding — this is precisely the standard senior interviewers apply when they
ask a follow-up like "show me" instead of accepting the first sentence.


---
# CHAPTER 52 — NumPy Coding Interview

For each problem: **attempt it yourself first** (cover the solution cell below it). Solutions
are provided for self-review, evaluated the way a senior engineer would: correctness,
complexity, memory, NumPy idiom quality, readability, production-readiness.

## Problem 1 — Rolling maximum, no loops
**Task:** Given a 1D array and a window size `w`, return the maximum value in every window of
size `w` (output length `n - w + 1`), without an explicit Python loop over window positions.
**Constraints:** `w <= len(array)`.
**Example:** `rolling_max([1,3,2,5,4,6], 3)` → `[3,5,5,6]`


In [ ]:
# --- Attempt Problem 1 yourself here before viewing the solution below ---
def rolling_max(arr, w):
    pass  # YOUR IMPLEMENTATION HERE


### Solution 1


In [ ]:
def rolling_max(arr, w):
    windows = np.lib.stride_tricks.sliding_window_view(arr, w)   # Chapter 48's zero-copy trick
    return windows.max(axis=1)                                     # Chapter 10's axis reduction

test = np.array([1, 3, 2, 5, 4, 6])
result = rolling_max(test, 3)
print("rolling_max([1,3,2,5,4,6], 3) =", result)
assert np.array_equal(result, [3, 5, 5, 6]), "FAILED"
print("PASSED")


**Evaluation notes:** `sliding_window_view` + `.max(axis=1)` is O(n) (no window-recomputation
waste, Chapter 48), uses no extra memory for the windows themselves (they're a stride-based
view, Chapter 28), and reads as a two-line, idiomatic solution — this is close to the "senior
engineer" answer. A correct-but-weaker solution using a Python loop calling `.max()` per window
would still pass correctness but should be flagged for avoidable `O(n×w)` complexity.

---

## Problem 2 — Vectorized pairwise Euclidean distance matrix
**Task:** Given `X` of shape `(n, d)`, return the `(n, n)` matrix of pairwise Euclidean
distances between all rows, without any Python-level loop.
**Constraints:** should work for any `n`, `d`.
**Example:** for 2 points at `(0,0)` and `(3,4)`, the distance is `5.0`.


In [ ]:
# --- Attempt Problem 2 yourself here ---
def pairwise_distances(X):
    pass  # YOUR IMPLEMENTATION HERE


### Solution 2


In [ ]:
def pairwise_distances(X):
    diffs = X[:, np.newaxis, :] - X[np.newaxis, :, :]   # Chapter 40/41's broadcasting trick, (n,n,d)
    return np.linalg.norm(diffs, axis=2)                 # Chapter 23's axis-aware norm, (n,n)

test = np.array([[0., 0.], [3., 4.], [0., 0.]])
result = pairwise_distances(test)
print("pairwise_distances:\n", result)
assert np.isclose(result[0, 1], 5.0), "FAILED"
assert np.allclose(np.diag(result), 0.0), "FAILED (diagonal should be all zeros)"
print("PASSED")


**Evaluation notes:** Correct and clean, directly reusing the K-Means/KNN broadcasting pattern
(Chapter 40/41) — this reuse is itself a signal of understanding, not coincidence. **Memory
caveat a senior reviewer would raise:** this allocates an intermediate `(n, n, d)` array, which
for large `n` and `d` can be substantial (Chapter 29's "vectorization can increase memory"
lesson, directly applicable) — a production-grade alternative uses the identity `||a-b||² =
||a||² + ||b||² - 2·a·b` to compute the same result via one `(n,d)@(d,n)` matmul plus two
`(n,)` norm vectors, avoiding the `(n,n,d)` intermediate entirely, at the cost of a small
numerical-stability trade-off (Chapter 27) for near-duplicate points.

---

## Problem 3 — Debug and fix
**Task:** This function is supposed to standardize each column of `X` (Chapter 36), but has a
bug. Find it, fix it, and explain why it's wrong.
```python
def standardize_buggy(X):
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    return (X - mean) / std
```


In [ ]:
# --- Attempt Problem 3 yourself: identify AND fix the bug before running this cell ---
def standardize_buggy(X):
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    return (X - mean) / std

X_test = np.array([[1., 2., 3.], [4., 5., 6.]])
try:
    result = standardize_buggy(X_test)
    print("Ran without error -- but is it silently wrong? Check the axis!", result)
except ValueError as e:
    print("Raised an error (shape mismatch surfaced it this time):", e)


### Solution 3


In [ ]:
def standardize_fixed(X):
    mean = X.mean(axis=0, keepdims=True)    # axis=0: per-COLUMN (per-feature) statistic, Chapter 10
    std = X.std(axis=0, keepdims=True)       # keepdims=True: correct broadcast alignment, Chapter 10/11
    std = np.where(std == 0, 1, std)          # guard against a constant column, Chapter 36
    return (X - mean) / std

result = standardize_fixed(X_test)
print("Fixed result:\n", result)
print("Per-column mean now ~0:", np.round(result.mean(axis=0), 10))
print("Per-column std now ~1 :", np.round(result.std(axis=0), 10))


**The bug, precisely:** `axis=1` collapses **columns** (computing one statistic per **row**),
the opposite of "standardize each column." For `X_test`'s shape `(2,3)`, `mean`/`std` end up
shape `(2,)` — this happens to broadcast against `X` (which is `(2,3)`) in the *wrong* way
(matching against the wrong trailing axis without `keepdims`), either raising an error or
(worse, for other shapes) silently computing something that looks plausible but is wrong —
exactly Chapter 10's `keepdims` lesson and this chapter's Case 6. The fix: `axis=0` (collapse
samples, keep one stat per feature) plus `keepdims=True` for correct broadcasting, plus a
zero-guard (Chapter 36).

---

## Problem 4 — Efficient top-K with original indices preserved
**Task:** Given `scores` (confidence values) and `ids` (parallel array of item identifiers),
return the `ids` of the top-K highest-scoring items, in descending order, without a full sort
of all scores.
**Example:** `top_k_ids([50, 90, 10, 70], ['a','b','c','d'], k=2)` → `['b', 'd']`


In [ ]:
# --- Attempt Problem 4 yourself here ---
def top_k_ids(scores, ids, k):
    pass  # YOUR IMPLEMENTATION HERE


### Solution 4


In [ ]:
def top_k_ids(scores, ids, k):
    scores = np.asarray(scores)
    ids = np.asarray(ids)
    top_idx = np.argpartition(scores, -k)[-k:]                    # Chapter 16: O(n) average top-K
    top_idx_sorted = top_idx[np.argsort(scores[top_idx])[::-1]]     # Chapter 16: sort only the K results
    return ids[top_idx_sorted]

result = top_k_ids([50, 90, 10, 70], ['a', 'b', 'c', 'd'], k=2)
print("top_k_ids result:", result)
assert list(result) == ['b', 'd'], "FAILED"
print("PASSED")


**Evaluation notes:** Correct use of `argpartition` (not a full `argsort` of all scores) for
the initial selection, then sorting only the K survivors — directly the Chapter 16/41/47/49
pattern reused for the fourth time in this course, which is exactly the point: recognizing
"this is a top-K problem" should now trigger this specific two-step idiom automatically, not a
from-scratch derivation each time.

---

## Chapter 50–52 Summary

- **Debugging**: errors split into loud (shape/matmul mismatches — Chapters 11, 22) and silent
  (dtype truncation, NaN propagation, view mutation, wrong axis — Chapters 4, 8, 15, 10) — the
  silent category is more dangerous precisely because nothing prompts you to look; the
  Chapter-3/4/8/15 diagnostic habits are the only defense.
- **Interview answers**: structure as Definition → Mechanism → Example → Trade-off → Real-world
  use (or Problem → Diagnosis → Solution → Trade-offs → Production consideration for system
  questions); expert-level answers benefit from a live demonstration, not just a recited
  principle.
- **Coding interview**: four problems solved with patterns this course already built —
  `sliding_window_view`, broadcasting-based pairwise distance, `axis`/`keepdims` debugging, and
  `argpartition`-based top-K — reinforcing that interview coding fluency comes from pattern
  recognition across a small set of core idioms, not memorizing individual problems.

---


